# Notebook 01: Formulación del problema y descripción de los datos

---

## 1.1 Análisis estratégico con enfoque Google

**Resultado ideal (objetivo del cliente).**
Un hotel quiere dejar de perder ingresos por habitaciones que quedan vacías sin posibilidad de
revenderse. Hoy trata a todas las reservas bajo las mismas condiciones, sin distinguir cuáles tienen
riesgo de no concretarse. El resultado ideal es poder decidir, al momento de recibir cada reserva,
si conviene exigir depósito, pedir garantía de tarjeta o aceptarla sin condiciones.

**Objetivo del modelo (acción concreta a predecir).**
Predecir en cuál de tres estados terminará una reserva (`Check-Out`, `Canceled` o `No-Show`)
usando únicamente información disponible **en el momento de reservar**.

**Filtro de formulación: por qué clasificación multiclase y no binaria.**
Cancelar y no presentarse parecen lo mismo, porque en ambos casos el huésped no se hospeda.
Para el negocio son situaciones distintas:

| Estado | Qué pasa | Acción comercial |
|---|---|---|
| Check-Out | El huésped se hospeda | Ninguna |
| Canceled | Avisa con anticipación | La habitación se revende |
| No-Show | No avisa y no llega | La noche se pierde entera |

Un modelo binario perdería exactamente la distinción que determina la política a aplicar.
La acción de negocio depende de *cuál* de las dos formas de no-hospedaje ocurre, por lo tanto el
problema exige tres clases.

**Métricas de éxito (definidas antes de tocar los datos).**
La clase `No-Show` es la más cara y la más rara. Un modelo que nunca la predijera acertaría cerca
del 99 % de los casos y sería inútil, porque justamente esa es la clase que cuesta dinero.
La métrica contractual es el **F1 macro**, que pondera las tres clases por igual, acompañada del
**recall de la clase No-Show**. La *accuracy* se reporta solo como referencia.

**Herramientas.** Python, pandas, NumPy, scikit-learn, matplotlib y seaborn, sobre Google Colab.

## 1.2 Objetivos del experimento

**Objetivo general.**
Construir y evaluar un clasificador multiclase basado en regresión logística que prediga el estado
final de una reserva hotelera a partir de las condiciones conocidas al momento de reservar.

**Objetivos específicos.**

1. Caracterizar el conjunto de datos, describiendo las variables predictoras y la distribución de
   las tres clases de la variable objetivo.
2. Diagnosticar y tratar los problemas de calidad del conjunto (nulos, duplicados, outliers e
   inconsistencias), documentando cada decisión.
3. Cuantificar el desbalance entre clases y aplicar una estrategia que evite que el modelo ignore
   la clase minoritaria.
4. Entrenar regresión logística en sus variantes Softmax y One-vs-Rest, explorando el hiperparámetro
   de regularización `C`, y seleccionar la configuración con mejor F1 macro sobre el test.
5. Comparar el desempeño en entrenamiento y prueba para diagnosticar sobreajuste, y analizar los
   errores por clase mediante la matriz de confusión.

## Carga del conjunto de datos

Proviene de Antonio, Almeida y Nunes (2019), *Hotel booking demand datasets*, publicado en
**Data in Brief**. Reúne reservas de un hotel urbano y uno vacacional en Portugal entre julio de
2015 y agosto de 2017. Se lee desde una URL pública: no requiere descarga ni credenciales.

### Entorno y persistencia de resultados

La celda siguiente fija tres cosas que condicionan la reproducibilidad del experimento:

**Semilla fija.** `RANDOM_STATE = 42` se aplica a la partición train/test y al ajuste de todos los
modelos. Sin ella, cada ejecución produciría particiones distintas y las métricas no serían
comparables entre corridas ni verificables por un tercero.

**Persistencia en Google Drive.** Los resultados se escriben en `MyDrive/hotel_booking` y no en el
disco temporal de Colab, que se borra al cerrar la sesión. Esto permite que el experimento se ejecute
en varias sesiones sin repetir etapas: el notebook 03 deja las particiones preparadas y los notebooks
04 y 05 las consumen tal cual, garantizando que todos operan exactamente sobre los mismos datos.
Si el montaje no se completa, la celda interrumpe la ejecución en lugar de escribir en una ubicación
volátil.

**Registro de las figuras.** La función `guardar` escribe cada gráfico en `splits/` como PNG a 200
dpi. Se invoca siempre antes de `plt.show()`, porque mostrar la figura vacía el buffer de matplotlib
y el archivo resultante quedaría en blanco.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

EN_DRIVE = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError(
            "Drive no quedo montado. Volve a ejecutar esta celda y autoriza el acceso "
            "en la ventana emergente."
        )
    RUTA = "/content/drive/MyDrive/hotel_booking"
    EN_DRIVE = True
except ImportError:
    RUTA = os.path.abspath("./hotel_booking")

CARPETA_SPLITS = os.path.join(RUTA, "splits")
os.makedirs(CARPETA_SPLITS, exist_ok=True)

def guardar(nombre):
    destino = os.path.join(CARPETA_SPLITS, nombre + ".png")
    plt.savefig(destino, dpi=200, bbox_inches="tight", facecolor="white")
    print("Grafico guardado:", destino)

print("Guardando en Google Drive" if EN_DRIVE else "Google Drive no disponible: guardando local")
print("Carpeta de trabajo:", RUTA)
print("Graficos (splits) :", CARPETA_SPLITS)

In [ ]:
URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
df = pd.read_csv(URL)
print("Filas:", df.shape[0], "| Columnas:", df.shape[1])

In [ ]:
df.head()

## 2.1 Descripción de variables

**Variable objetivo:** `reservation_status`, con tres clases (K = 3).

Las variables predictoras se agrupan en cinco familias:

| Familia | Variables | Tipo | Significado en el dominio |
|---|---|---|---|
| Anticipación y fechas | `lead_time`, `arrival_date_year`, `arrival_date_month`, `arrival_date_week_number` | Numérica / categórica | Días entre reserva y llegada, y temporada |
| Composición de la estadía | `stays_in_weekend_nights`, `stays_in_week_nights`, `adults`, `children`, `babies` | Numérica discreta | Duración y número de huéspedes |
| Canal y condiciones | `hotel`, `meal`, `market_segment`, `distribution_channel`, `deposit_type`, `customer_type`, `reserved_room_type`, `country`, `agent` | Categórica nominal | Cómo y por dónde entró la reserva |
| Historial del cliente | `is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled`, `booking_changes`, `days_in_waiting_list`, `required_car_parking_spaces`, `total_of_special_requests` | Numérica discreta | Comportamiento previo y exigencias |
| Precio | `adr` | Numérica continua | Tarifa diaria promedio, en euros |

In [ ]:
print("=== Tipos de dato ===")
print(df.dtypes.value_counts(), "\n")

cat_cols_raw = df.select_dtypes(include="object").columns.tolist()
print("=== Cardinalidad de las variables categoricas ===")
print(df[cat_cols_raw].nunique().sort_values(ascending=False), "\n")

print("=== Variable objetivo ===")
print(df["reservation_status"].value_counts())

**Hallazgo que condiciona el preprocesamiento.** `country` tiene 178 categorías y `agent` más de 300.
Codificarlas directamente con one-hot generaría cientos de columnas, la mayoría con muy pocas
observaciones. El notebook 03 documenta cómo se reducen.
